# Heart Disease Prediction: Linear Regression vs Decision Tree (ID3)
**ML Class Project (SRM-AP)**

Dataset: Kaggle *Heart Disease Dataset* (`johnsmith88/heart-disease-dataset`, file `heart.csv`), the same dataset used in the reference notebook.

Pipeline: load, verify the dataset, EDA (matplotlib), 80/20 train-test split, preprocessing, Linear Regression, Decision Tree, evaluation and comparison.

**Note on Linear Regression:** the target is binary (0 = no disease, 1 = disease), so Linear Regression is used here as a *linear probability model*. It predicts a continuous score, and a score of 0.5 or more is classified as disease. We report regression metrics (MAE, RMSE, R2) and classification metrics (accuracy, precision, recall, F1, ROC-AUC).

## Step 0: Imports

In [ ]:
!pip install -q kagglehub

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

## Step 1: Load the dataset (same Kaggle dataset as the reference notebook)

In [ ]:
try:
    import kagglehub
    path = kagglehub.dataset_download("johnsmith88/heart-disease-dataset")
    df = pd.read_csv(os.path.join(path, "heart.csv"))
    print("Loaded from Kaggle:", path)
except Exception as e:
    print("Kaggle download failed:", e)
    print("Download heart.csv manually from Kaggle and upload it below.")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(list(uploaded.keys())[0])

print("Raw shape:", df.shape)
df.head()

## Step 2: Dataset verification
Each check prints PASS or WARN. A WARN does not stop the notebook, but read it before trusting the results. The expected values come from the attribute list and counts in the reference notebook (14 columns, ages 29 to 77, 302 records after duplicate removal).

In [ ]:
EXPECTED_COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
                 "exang","oldpeak","slope","ca","thal","target"]
checks = []
def check(name, ok, detail=""):
    checks.append({"Check": name, "Result": "PASS" if ok else "WARN", "Detail": detail})

def in_range(col, lo, hi):
    return col in df.columns and df[col].between(lo, hi).all()

# --- Checks on the raw file ---
check("Has the 14 expected columns", set(EXPECTED_COLS) == set(df.columns),
      f"missing={set(EXPECTED_COLS)-set(df.columns)}, extra={set(df.columns)-set(EXPECTED_COLS)}")
check("All columns numeric", all(pd.api.types.is_numeric_dtype(t) for t in df.dtypes))
check("No missing values", df.isnull().sum().sum() == 0, f"total NaN = {df.isnull().sum().sum()}")
check("Target is binary (0/1)", set(df["target"].unique()) <= {0, 1}, f"values = {sorted(int(v) for v in df['target'].unique())}")
for c in ["sex", "fbs", "exang"]:
    check(f"{c} is binary (0/1)", in_range(c, 0, 1))
check("cp in 0..3", in_range("cp", 0, 3))
check("restecg in 0..2", in_range("restecg", 0, 2))
check("slope in 0..2", in_range("slope", 0, 2))
check("ca in 0..4", in_range("ca", 0, 4))
check("thal in 0..3", in_range("thal", 0, 3))
check("age within 29..77 (reference notebook)", in_range("age", 29, 77),
      f"min={df['age'].min()}, max={df['age'].max()}")
check("trestbps / chol / thalach all positive",
      bool((df[["trestbps","chol","thalach"]] > 0).all().all()))

n_dups = int(df.duplicated().sum())
print(f"Duplicate rows in raw file: {n_dups}")

# --- Remove duplicates (done BEFORE splitting so identical rows cannot land in both train and test) ---
df = df.drop_duplicates().reset_index(drop=True)

# --- Checks after cleaning ---
check("Shape after dedupe is (302, 14) (reference notebook)", df.shape == (302, 14), f"shape = {df.shape}")
share = df["target"].value_counts(normalize=True)
check("Classes reasonably balanced (smaller class >= 40%)", share.min() >= 0.40,
      f"disease={share.get(1,0):.1%}, no disease={share.get(0,0):.1%}")

report = pd.DataFrame(checks)
print(report.to_string(index=False))
print("\nSummary:", (report["Result"] == "PASS").sum(), "PASS /", (report["Result"] == "WARN").sum(), "WARN")

## Step 3: Visualize the dataset (matplotlib)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df["target"].value_counts().sort_index()
axes[0].bar(["No disease (0)", "Disease (1)"], counts.values, color=["#4c9a8b", "#c8553d"], edgecolor="black")
axes[0].set_title("Class Balance")
axes[0].set_ylabel("Number of Records")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha="center")

axes[1].hist(df["age"], bins=10, edgecolor="black", color="#4c9a8b")
axes[1].set_title("Distribution of Patient Ages")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Number of Records")
plt.tight_layout()
plt.show()

In [ ]:
features = ["trestbps", "chol", "thalach", "oldpeak"]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, f in zip(axes.ravel(), features):
    ax.hist(df[f], bins=15, edgecolor="black", color="#4c9a8b")
    ax.set_title(f)
plt.tight_layout()
plt.show()

In [ ]:
# Feature values split by class
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, f in zip(axes, ["thalach", "oldpeak", "age"]):
    ax.boxplot([df.loc[df["target"] == 0, f], df.loc[df["target"] == 1, f]])
    ax.set_xticklabels(["No disease", "Disease"])
    ax.set_title(f + " by class")
plt.tight_layout()
plt.show()

# Age vs max heart rate, coloured by class
plt.figure(figsize=(7, 5))
for t, colour, label in [(0, "#4c9a8b", "No disease"), (1, "#c8553d", "Disease")]:
    sub = df[df["target"] == t]
    plt.scatter(sub["age"], sub["thalach"], c=colour, label=label, alpha=0.7, edgecolor="black", linewidth=0.3)
plt.xlabel("Age")
plt.ylabel("Max heart rate (thalach)")
plt.title("Age vs Max Heart Rate")
plt.legend()
plt.show()

In [ ]:
# Correlation heatmap (matplotlib only)
cols = ["age", "trestbps", "chol", "thalach", "oldpeak", "target"]
corr = df[cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## Step 4: Train-test split (80/20, stratified)

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training features:", X_train.shape, "| Testing features:", X_test.shape)
print("Disease share  train: %.3f   test: %.3f" % (y_train.mean(), y_test.mean()))

## Step 5: Preprocessing
Numeric columns are scaled, multi-category columns are one-hot encoded, and binary columns pass through. The preprocessor lives inside each model pipeline, so it is fit on training data only and the test set stays unseen.
Note: this preprocessor is used by Linear Regression only. The ID3 tree in Step 7 works on discrete attributes, so it bins the numeric columns itself.

In [ ]:
numeric_attributes = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_attributes = ["cp", "restecg", "slope", "ca", "thal"]
binary_attributes = ["sex", "fbs", "exang"]

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("numerical", StandardScaler(), numeric_attributes),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_attributes),
        ("binary", "passthrough", binary_attributes),
    ])

## Step 6: Model 1, Linear Regression
Predicts a continuous disease score. A score of 0.5 or more is classified as disease.

In [ ]:
lin_pipe = make_pipeline(make_preprocessor(), LinearRegression())
lin_pipe.fit(X_train, y_train)

lin_score = lin_pipe.predict(X_test)            # continuous score
lin_pred = (lin_score >= 0.5).astype(int)       # thresholded class

print("Regression metrics (on the continuous score)")
print("  MAE :", round(mean_absolute_error(y_test, lin_score), 4))
print("  RMSE:", round(float(np.sqrt(mean_squared_error(y_test, lin_score))), 4))
print("  R2  :", round(r2_score(y_test, lin_score), 4))

In [ ]:
# Where does the score fall for each class, and where is the 0.5 threshold?
plt.figure(figsize=(8, 4.5))
plt.hist(lin_score[y_test == 0], bins=15, alpha=0.7, label="No disease", color="#4c9a8b", edgecolor="black")
plt.hist(lin_score[y_test == 1], bins=15, alpha=0.7, label="Disease", color="#c8553d", edgecolor="black")
plt.axvline(0.5, color="black", linestyle="--", label="Threshold 0.5")
plt.xlabel("Linear Regression score")
plt.ylabel("Number of test patients")
plt.title("Linear Regression Scores by True Class")
plt.legend()
plt.show()

# Largest coefficients
feat_names = lin_pipe[:-1].get_feature_names_out()
coefs = pd.Series(lin_pipe[-1].coef_, index=feat_names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(10)[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top.index, top.values, color=["#c8553d" if v > 0 else "#4c9a8b" for v in top.values], edgecolor="black")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Linear Regression: Top 10 Coefficients")
plt.tight_layout()
plt.show()

## Step 7: Model 2, Decision Tree using the ID3 algorithm
ID3 (Iterative Dichotomiser 3) is written from scratch here, not taken from scikit-learn. It builds the tree top-down:

1. **Entropy** measures how mixed the labels are: `H(S) = -sum(p * log2(p))` over the classes. A pure group has entropy 0; a 50/50 group has entropy 1.
2. **Information gain** of an attribute A is the drop in entropy after splitting on it: `Gain(S, A) = H(S) - sum(|Sv|/|S| * H(Sv))`, summed over each value v of A.
3. At each node, ID3 picks the attribute with the **highest information gain**, creates **one branch per value** of that attribute, and recurses on each branch with that attribute removed.
4. It stops when a node is pure, no attributes are left, or (our one addition) a depth limit is reached. A leaf predicts the majority class and reports P(disease) as the share of disease patients in the leaf.

**Differences from scikit-learn's tree (CART):** ID3 uses entropy only, splits into one branch per value (not two), and works on **discrete attributes only**. So the five numeric columns are cut into quartile bins first (bin edges learned from the training data only). The original ID3 has no pruning and overfits, so the depth limit is chosen by 5-fold cross-validation on the training data.

In [ ]:
class QuantileBinner:
    """ID3 needs discrete attributes, so numeric columns are cut into quartile bins.
    Edges are learned from the training data only, then reused on the test data."""
    def __init__(self, columns, n_bins=4):
        self.columns, self.n_bins = columns, n_bins
    def fit(self, X):
        qs = np.linspace(0, 1, self.n_bins + 1)[1:-1]
        self.edges_ = {c: np.unique(np.quantile(X[c], qs)) for c in self.columns}
        return self
    def transform(self, X):
        X = X.copy()
        for c, edges in self.edges_.items():
            X[c] = np.digitize(X[c], edges)      # bin index 0, 1, 2, ...
        return X


def entropy(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))


def information_gain(x, y):
    """Gain(S, A) = H(S) - sum over values v of |Sv|/|S| * H(Sv)"""
    weighted = 0.0
    for v in np.unique(x):
        mask = (x == v)
        weighted += mask.mean() * entropy(y[mask])
    return entropy(y) - weighted


class ID3Classifier:
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split

    def fit(self, X, y):
        self.tree_ = self._build(X.reset_index(drop=True), np.asarray(y), list(X.columns), 0)
        return self

    def _leaf(self, y):
        p1 = float(np.mean(y))
        return {"leaf": True, "p1": p1, "n": len(y), "label": int(p1 >= 0.5)}

    def _build(self, X, y, attrs, depth):
        # Stop: pure node, no attributes left, too few samples, or depth limit reached
        if (len(np.unique(y)) == 1 or not attrs or len(y) < self.min_samples_split
                or (self.max_depth is not None and depth >= self.max_depth)):
            return self._leaf(y)
        gains = {a: information_gain(X[a].values, y) for a in attrs}
        best = max(gains, key=gains.get)
        if gains[best] <= 0:
            return self._leaf(y)
        node = {"leaf": False, "attr": best, "gain": gains[best],
                "p1": float(np.mean(y)), "n": len(y), "children": {}}
        remaining = [a for a in attrs if a != best]      # an attribute is used once per path
        for v in np.unique(X[best].values):               # one branch per value
            m = (X[best] == v).values
            node["children"][v] = self._build(X[m].reset_index(drop=True), y[m], remaining, depth + 1)
        return node

    def _walk(self, row):
        node = self.tree_
        while not node["leaf"]:
            child = node["children"].get(row[node["attr"]])
            if child is None:            # value never seen at this node during training
                return node["p1"]
            node = child
        return node["p1"]

    def predict_proba(self, X):
        return np.array([self._walk(r) for r in X.to_dict("records")])

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


def tree_depth(node):
    return 0 if node["leaf"] else 1 + max(tree_depth(c) for c in node["children"].values())

def count_leaves(node):
    return 1 if node["leaf"] else sum(count_leaves(c) for c in node["children"].values())

def print_tree(node, indent="", max_show=3, depth=0):
    if node["leaf"]:
        print(f"{indent}-> {'DISEASE' if node['label'] else 'NO DISEASE'} (n={node['n']}, P(disease)={node['p1']:.2f})")
        return
    if depth >= max_show:
        print(f"{indent}... (subtree continues, {count_leaves(node)} leaves)")
        return
    print(f"{indent}[{node['attr']}]  gain={node['gain']:.3f}, n={node['n']}")
    for v, child in node["children"].items():
        print(f"{indent}  {node['attr']} = {v}:")
        print_tree(child, indent + "    ", max_show, depth + 1)

In [ ]:
def fit_id3(X_tr, y_tr, **kw):
    binner = QuantileBinner(numeric_attributes).fit(X_tr)
    return binner, ID3Classifier(**kw).fit(binner.transform(X_tr), y_tr)

def accuracy_of(binner, model, X, y):
    return float((model.predict(binner.transform(X)) == np.asarray(y)).mean())

# Baseline: pure ID3 with no limits
b0, id3_full = fit_id3(X_train, y_train)
print("Pure ID3 (no limit): depth %d, %d leaves" % (tree_depth(id3_full.tree_), count_leaves(id3_full.tree_)))
print("  train acc: %.3f | test acc: %.3f" % (accuracy_of(b0, id3_full, X_train, y_train),
                                              accuracy_of(b0, id3_full, X_test, y_test)))

# Choose max_depth by 5-fold CV on the TRAINING data only (binning is re-fit inside every fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
depths = [1, 2, 3, 4, 5, 6, 8, None]
cv_scores = {}
for d in depths:
    accs = []
    for tr_idx, va_idx in skf.split(X_train, y_train):
        b, m = fit_id3(X_train.iloc[tr_idx], y_train.iloc[tr_idx], max_depth=d)
        accs.append(accuracy_of(b, m, X_train.iloc[va_idx], y_train.iloc[va_idx]))
    cv_scores[d] = np.mean(accs)
    print("max_depth =", d, "-> CV accuracy: %.3f" % cv_scores[d])

best_depth = max(cv_scores, key=cv_scores.get)
print("\nBest max_depth:", best_depth)

In [ ]:
# Final ID3 tree with the chosen depth
binner, id3 = fit_id3(X_train, y_train, max_depth=best_depth)
X_train_b, X_test_b = binner.transform(X_train), binner.transform(X_test)

tree_pred = id3.predict(X_test_b)
tree_proba = id3.predict_proba(X_test_b)          # P(disease) = share of disease patients in the leaf
print("ID3 (depth limit %s): depth %d, %d leaves" % (best_depth, tree_depth(id3.tree_), count_leaves(id3.tree_)))
print("  train acc: %.3f | test acc: %.3f" % (accuracy_of(binner, id3, X_train, y_train),
                                              accuracy_of(binner, id3, X_test, y_test)))

# Information gain of every attribute at the root: the largest one is what ID3 picks first
print("\nEntropy of the training labels H(S) = %.3f bits" % entropy(y_train.values))
root_gain = pd.Series({a: information_gain(X_train_b[a].values, y_train.values)
                       for a in X_train_b.columns}).sort_values()
print("Root attribute chosen:", id3.tree_["attr"])
plt.figure(figsize=(8, 5))
plt.barh(root_gain.index, root_gain.values, color="#4c9a8b", edgecolor="black")
plt.xlabel("Information gain (bits)")
plt.title("ID3: Information Gain of Each Attribute at the Root")
plt.tight_layout()
plt.show()

# CV accuracy vs depth limit
plt.figure(figsize=(7, 4.5))
plt.plot(range(len(depths)), list(cv_scores.values()), marker="o", color="#4c9a8b")
plt.xticks(range(len(depths)), [str(d) if d is not None else "None" for d in depths])
plt.xlabel("max_depth")
plt.ylabel("5-fold CV accuracy (training data)")
plt.title("ID3: Accuracy vs Depth Limit")
plt.grid(alpha=0.3)
plt.show()

# The learned rules (top 3 levels)
print_tree(id3.tree_, max_show=3)

## Step 8: Evaluation and comparison on the held-out test set

In [ ]:
def class_metrics(y_true, y_pred, score):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, score),
    }

comparison = pd.DataFrame({
    "Linear Regression (threshold 0.5)": class_metrics(y_test, lin_pred, lin_score),
    "Decision Tree (ID3)": class_metrics(y_test, tree_pred, tree_proba),
}).T.round(3)
print(comparison)

# Grouped bar chart
ax = comparison.plot(kind="bar", figsize=(10, 5), edgecolor="black", color=["#4c9a8b", "#c8553d", "#e0a93b", "#5b7db1", "#8a8a8a"])
ax.set_ylim(0, 1.05)
ax.set_title("Model Comparison on Test Set")
ax.set_ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, pred) in zip(axes, [("Linear Regression", lin_pred), ("Decision Tree (ID3)", tree_pred)]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=["No disease", "Disease"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

# ROC curves
plt.figure(figsize=(6, 5))
for name, score in [("Linear Regression", lin_score), ("Decision Tree (ID3)", tree_proba)]:
    fpr, tpr, _ = roc_curve(y_test, score)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, score):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.show()

## Step 9: Conclusion
Fill this in with your own numbers from Step 8:
- Which model scored higher on accuracy, recall and F1, and by how much?
- Recall matters most in a screening setting, since a missed disease case is costlier than a false alarm. Which model missed fewer?
- Which attribute did ID3 pick at the root (see the information gain chart), and which features did the linear coefficients rely on most (the reference notebook found `oldpeak` and `thalach` most correlated with the target)?
- The test set has only about 61 patients, so one or two predictions change the percentages noticeably. Treat small gaps between models with caution.